# Part 1 - Multiple Choice Queries

This notebook answers the 10 MCQ questions directly from the provided CSV files in `../Raw Data`. Run it from this folder after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import duckdb

RAW_DIR = (Path.cwd().parent / "Raw Data").resolve()
assert RAW_DIR.exists(), f"Raw data folder not found: {RAW_DIR}"

con = duckdb.connect()
for file in sorted(RAW_DIR.glob("*.csv")):
    table_name = file.stem
    csv_path = file.as_posix().replace("'", "''")
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{csv_path}', header=True)
    """)

print(f"Loaded CSV tables from {RAW_DIR}")
print(con.execute("SHOW TABLES").df())


# **Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)**

In [ ]:
query = """
WITH ranking_table AS (
    SELECT
        customer_id,
        order_date,
        LEAD(order_date) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS following_date
    FROM orders
    WHERE customer_id IN (
        SELECT customer_id
        FROM orders
        GROUP BY customer_id
        HAVING COUNT(*) > 1
    )
),

filtered_table AS (
    SELECT *
    FROM ranking_table
    WHERE following_date IS NOT NULL
),

gap_table AS (
    SELECT
        customer_id,
        DATE_DIFF('day', order_date, following_date) AS interorder_gap
    FROM filtered_table
)

SELECT
    PERCENTILE_CONT(0.5)
    WITHIN GROUP (ORDER BY interorder_gap) AS median_gap
FROM gap_table;
"""

con.execute(query).df()

# **Q1. C. GAP = 144**

# **Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?**

In [ ]:
query = """
SELECT
    segment,
    AVG((price - cogs) * 1.0 / price) AS avg_profit_margin
FROM products
GROUP BY segment
ORDER BY avg_profit_margin DESC;
"""

con.execute(query).df()

# **Q2. D. STANDARD**

# **Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?**

In [ ]:
query = """
SELECT return_reason, COUNT(return_reason) AS return_frequency
FROM returns t1
LEFT JOIN products t2 ON t1.product_id = t2.product_id
WHERE Category = 'Streetwear'
GROUP BY return_reason
ORDER BY COUNT(return_reason) DESC;
"""

con.execute(query).df()

# **Q3. B. WRONG-SIZE**

# **Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?**

In [ ]:
query = """
SELECT traffic_source, AVG(bounce_rate) AS avg_bounce_rate
FROM web_traffic
GROUP BY traffic_source
ORDER BY AVG(bounce_rate) ASC
"""

con.execute(query).df()

# **Q4. C. EMAIL**

# **Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi  (tức là promo_id không null) xấp xỉ là bao nhiêu?**

In [ ]:
query = """
SELECT SUM(CASE
    WHEN promo_id IS NOT NULL THEN 1
    ELSE 0 END
    )*1.00/COUNT(order_id) AS applied_promo_percent
FROM order_items
"""

con.execute(query).df()

# **Q5. C. 39%**

# **Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn/số khách hàng trong nhóm)**

In [ ]:
query = """
SELECT age_group, COUNT(order_id)*1.00/COUNT(DISTINCT T1.customer_id) AS AOV
FROM orders t1
LEFT JOIN customers t2 ON t1.customer_id = t2.customer_id
WHERE age_group IS NOT NULL
GROUP BY age_group
ORDER BY COUNT(order_id)*1.00/COUNT(DISTINCT T1.customer_id) DESC
"""

con.execute(query).df()

# **Q6. A. 55+**

# **Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?**

In [ ]:
query = """
SELECT region, SUM(unit_price*quantity-discount_amount) AS total_rev_by_region
FROM orders t1
LEFT JOIN order_items t2 ON t1.order_id = t2.order_id
LEFT JOIN geography t3 ON t1.zip = t3.zip
WHERE YEAR(t1.order_date) <2023
GROUP BY region
ORDER BY SUM(unit_price*quantity-discount_amount) DESC
"""

con.execute(query).df()

# **Q7. C. EAST**

# **Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv,  phương thức thanh toán nào được sử dụng nhiều nhất?**

In [ ]:
query = """
SELECT payment_method
    ,COUNT (order_id) AS num_payment_method
FROM orders
WHERE order_status = 'cancelled'
GROUP BY payment_method
ORDER BY COUNT (order_id) DESC
"""

con.execute(query).df()

# **Q8. A. CREDIT CARD**

# **Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?**

In [ ]:
query = """
SELECT size
    , SUM(
        CASE
            WHEN return_id IS NOT NULL THEN 1
            ELSE 0 END
    )*1.00/COUNT(T1.order_id) AS return_rate
FROM order_items t1
LEFT JOIN products t2 ON t1.product_id = t2.product_id
LEFT JOIN returns t3 ON t1.order_id=t3.order_id AND t1.product_id = t3.product_id
GROUP BY size
ORDER BY SUM(
        CASE
            WHEN return_id IS NOT NULL THEN 1
            ELSE 0 END
    )*1.00/COUNT(T1.order_id) DESC
"""

con.execute(query).df()

# **Q9. A. S**

# **Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?**

In [ ]:
query ="""
SELECT installments, AVG(payment_value) AS avg_payment_value_per_order
FROM payments
GROUP BY installments
ORDER BY AVG(payment_value) DESC
"""
con.execute(query).df()

# **Q10. C. 6**